# SSL400 ST-GCN — Google Colab Training

Train the baseline ST-GCN on MediaPipe Pose CSVs.

## Before you start

1. Upload to Google Drive:
   - `ssl400-research/` (this project)
   - `Dataset - MP - CSV/` (landmark CSVs)
2. Suggested Drive layout:
```text
MyDrive/SSL400/
  ssl400-research/
  Dataset - MP - CSV/
```
3. In Colab: **Runtime → Change runtime type → GPU (T4) → Save**
4. Run cells **in order**.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Go to project folder

Edit `PROJECT_DIR` / `DATASET_DIR` if your Drive paths differ.

In [ ]:
import os
from pathlib import Path

# --- EDIT THESE if your Drive folders are elsewhere ---
PROJECT_DIR = Path('/content/drive/MyDrive/SSL400/ssl400-research')
DATASET_DIR = Path('/content/drive/MyDrive/SSL400/Dataset - MP - CSV')
# -------------------------------------------------------

assert PROJECT_DIR.exists(), f'Missing project: {PROJECT_DIR}'
assert DATASET_DIR.exists(), f'Missing dataset: {DATASET_DIR}'

os.chdir(PROJECT_DIR)
print('CWD:', Path.cwd())
print('Project contents:', sorted(p.name for p in PROJECT_DIR.iterdir()))
print('Dataset categories (sample):', sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())[:8], '...')

## 3) Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Runtime → Change runtime type → GPU (T4)')

## 4) Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Install done')

## 5) Write Colab-friendly configs

Uses absolute Drive path for the dataset, enables AMP, and keeps `num_workers: 0` (safer with Google Drive).

In [ ]:
from pathlib import Path

data_yaml = f'''# SSL400 data config (Colab)
raw_csv_root: "{DATASET_DIR.as_posix()}"
processed_dir: "data/processed"
inspected_dir: "data/inspected"

min_samples_per_class: 10
sequence_length: 64
use_z: true
visibility_threshold: 0.5
mask_low_visibility: true

train_ratio: 0.70
val_ratio: 0.15
test_ratio: 0.15
split_seed: 42
save_npz: true
'''

model_yaml = '''# ST-GCN config (Colab GPU)
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 80
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  step_size: 30
  gamma: 0.1
  early_stopping_patience: 15
  num_workers: 0
  amp: true
  class_weights: true
  seed: 42
  device: auto

paths:
  checkpoint_dir: "checkpoints"
  results_dir: "results"
  experiment_name: "baseline_stgcn_seed42"
'''

Path('configs').mkdir(exist_ok=True)
Path('configs/data.yaml').write_text(data_yaml, encoding='utf-8')
Path('configs/model.yaml').write_text(model_yaml, encoding='utf-8')
print('Wrote configs/data.yaml')
print('Wrote configs/model.yaml')
print('raw_csv_root =', DATASET_DIR)

## 6) Prepare dataset (run once)

Skip this cell later if `data/processed/manifest.json` already exists.

In [ ]:
from pathlib import Path

manifest = Path('data/processed/manifest.json')
if manifest.exists():
    print('Processed data already exists — skipping prepare.')
    print('Delete data/processed/ and re-run this cell if you changed data.yaml.')
else:
    !python scripts/prepare_dataset.py --data-config configs/data.yaml

In [ ]:
import json
from pathlib import Path
import numpy as np

class_map = json.loads(Path('data/processed/class_to_idx.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('data/processed/manifest.json').read_text(encoding='utf-8'))
sample = manifest['train'][0]
tensor = np.load(sample['tensor_path'])

print('Classes:', len(class_map))
print('Split sizes:', {k: len(v) for k, v in manifest.items()})
print('Example:', sample['sample_id'], 'shape', tensor.shape, tensor.dtype)

## 7) Train ST-GCN

Keep this tab open. Checkpoints save under `checkpoints/` on Drive.

If you get **CUDA out of memory**, re-run cell 5 with `batch_size: 16` (or `8`), then train again.

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model.yaml

## 8) Evaluate on frozen test set

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint checkpoints/baseline_stgcn_seed42_best.pt \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model.yaml

## 9) Export trained model to `.h5`

Saves PyTorch weights + metadata into HDF5 (`checkpoints/baseline_stgcn_seed42_best.h5`).

Note: this is **not** a Keras `model.save()` file. Use `h5py` (or reload weights into the ST-GCN class) to read it.

In [ ]:
from pathlib import Path

ckpt_path = Path('checkpoints/baseline_stgcn_seed42_best.pt')
h5_path = Path('checkpoints/baseline_stgcn_seed42_best.h5')

assert ckpt_path.exists(), f'Train first — missing checkpoint: {ckpt_path}'

!python scripts/export_h5.py --checkpoint checkpoints/baseline_stgcn_seed42_best.pt --output checkpoints/baseline_stgcn_seed42_best.h5

# Quick verify
import h5py
with h5py.File(h5_path, 'r') as f:
    weight_keys = [k for k in f['model_weights'].keys() if k != 'param_names']
    print('H5 file:', h5_path.resolve())
    print('Size MB:', round(h5_path.stat().st_size / (1024 * 1024), 2))
    print('Num weight tensors:', len(weight_keys))
    print('num_classes:', int(f['metadata'].attrs['num_classes']))
    print('in_channels:', int(f['metadata'].attrs['in_channels']))
    print('Sample keys:', weight_keys[:5])

## 10) Show training summary / download `.pt` and `.h5`

In [ ]:
import json
from pathlib import Path
from google.colab import files

summary_path = Path('results/baseline_stgcn_seed42_train_summary.json')
metrics_path = Path('results/baseline_stgcn_seed42_test_metrics.json')
ckpt_path = Path('checkpoints/baseline_stgcn_seed42_best.pt')
h5_path = Path('checkpoints/baseline_stgcn_seed42_best.h5')

if summary_path.exists():
    print('Train summary:')
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    keep = {k: metrics[k] for k in metrics if k not in ('y_true', 'y_pred', 'sample_ids')}
    print('\nTest metrics:')
    print(json.dumps(keep, indent=2))

print('\nCheckpoint .pt:', ckpt_path.exists(), ckpt_path)
print('Export .h5:', h5_path.exists(), h5_path)

# Uncomment to download to your laptop:
# if ckpt_path.exists():
#     files.download(str(ckpt_path))
# if h5_path.exists():
#     files.download(str(h5_path))

## Notes

- Runtime: **GPU T4** (not CPU, not TPU).
- Dropout(0.5) is already in the model for training; MC Dropout *inference* is a later step.
- After training, section **9** exports `checkpoints/baseline_stgcn_seed42_best.h5` (PyTorch weights in HDF5).
- If Colab disconnects: remount Drive, `%cd` to project, skip prepare if done, run train again.
- Outputs stay on Drive under `ssl400-research/checkpoints` and `results`.